# Kongo MT Challenge - T5 Starter Notebook

**IndabaX DRC 2026: Kongo MT Challenge.** Build **one** multilingual model that translates
between **French (`fr`)** and **five low-resource Congolese languages**:

| Code | Language | Train pairs / direction |
|------|----------|-------------------------|
| `hav` | Havu (Kihavu)     | 2,593  |
| `ngc` | Ngombe (Lingombe) | 10,263 |
| `shr` | Shi (Mashi)       | 24,720 |
| `tap` | Taabwa (Kitaabwa) | 4,048  |
| `yns` | Yansi (Kiyansi)   | 6,332  |

Translation goes **both ways** - French into each language and each language into French -
so there are **10 directions**, all scored on one leaderboard by a **single** model. Every row
carries its own `SOURCE_LANG` and `TARGET_LANG`; this notebook reads them per row, so the same
code covers all 10 directions. The data is verse-aligned Bible text (one domain). It is
imbalanced - Shi has ~10x Havu's pairs - and **Shi and Havu are close relatives**, so training
all directions together lets the model transfer from higher- to lower-resource languages.

**Pipeline**
1. Load `Train.csv` / `Val.csv` / `Test.csv`
2. Build a HuggingFace dataset and tokenize (with a per-row `translate X to Y:` prefix)
3. Fine-tune with `Seq2SeqTrainer`, tracking the **ROUGE F-measure** locally
4. Translate `Test.csv` and write the submission (`CANDIDATE_TEXT`) straight from it

**Official scoring (Zindi).** The leaderboard uses **`Score = 0.1 * BLEU`** (0-100 scale,
**whitespace tokenization**), averaged over all 10 directions. Public leaderboard = 3,550 test
rows; final private ranking = the remaining 8,354. This notebook tracks the **ROUGE F-measure**
during training as its local signal; keep the official BLEU in mind when comparing submissions.

**Base model.** We default to **`t5-base`** to get a working baseline running quickly. Since plain
T5 was pretrained mostly on English, multilingual bases usually do better on these languages -
consider **mT5** or **ByT5** (Google), or dedicated MT models like **NLLB-200** / **M2M-100**.
Switch by changing `MODEL_NAME`.

> Run on a GPU (Colab/Kaggle: Runtime -> GPU). On CPU, use the small-subset switch in the data cell.

In [ ]:
# Install dependencies (Colab/Kaggle usually have most of these already)
!pip -q install "transformers>=4.41" "datasets>=2.19" rouge-score sentencepiece accelerate

In [ ]:
import os, warnings, logging
warnings.filterwarnings("ignore")                      # hide deprecation/user warnings
os.environ["TOKENIZERS_PARALLELISM"] = "false"         # silence the HF tokenizers fork warning
logging.getLogger("transformers").setLevel(logging.ERROR)

## 1. Imports and configuration

Tweak the values in the config block to trade off quality vs. training time.

In [ ]:
import os, numpy as np, pandas as pd, torch
from rouge_score import rouge_scorer
from datasets import Dataset
from transformers import (AutoTokenizer, AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq,
                          Seq2SeqTrainingArguments, Seq2SeqTrainer, set_seed)

# ---------------- Configuration ----------------
MODEL_NAME = "t5-base"            # simple baseline. Stronger alternatives (just swap the name in):
                                  #   "google/mt5-small" / "google/mt5-base"  -> multilingual T5
                                  #   "google/byt5-small"                     -> byte-level, good for Bantu morphology
                                  #   "facebook/nllb-200-distilled-600M"      -> dedicated low-resource MT
DATA_DIR        = "."             # folder containing the CSV files
OUT_DIR         = "t5-fr-target"  # where checkpoints are written
SUBMISSION_PATH = "submission.csv"

MAX_INPUT_LENGTH  = 128           # max source tokens (verses are short; raise if truncated)
MAX_TARGET_LENGTH = 128           # max target tokens
BATCH_SIZE        = 8
NUM_EPOCHS        = 8
LEARNING_RATE     = 5e-4          # mT5 likes a fairly high LR
NUM_BEAMS         = 4             # beam search width at generation time
SEED              = 42
# ------------------------------------------------

set_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE, "| Base model:", MODEL_NAME)

Device: cuda | Base model: t5-base


## 2. Load the data

- **Train.csv** (67,124 rows) / **Val.csv** (28,788 rows): `ID, SOURCE_LANG, TARGET_LANG,
  SOURCE_TEXT, TARGET_TEXT, VERSE_KEY`. You may train on Val as well once you are tuning.
- **Test.csv** (3,550 rows): `ID, SOURCE_LANG, SOURCE_TEXT, TARGET_LANG` - `TARGET_TEXT` is
  hidden; you predict it.
- **Submission**: same rows/IDs as `Test.csv`, columns `ID, SOURCE_LANG, SOURCE_TEXT,
  TARGET_LANG, CANDIDATE_TEXT`. Every row needs a non-empty prediction (blank rows score 0).

`VERSE_KEY` (e.g. `MAT.1.3`) is the verse reference, in Train/Val only. All text is UTF-8.

In [ ]:
train_df = pd.read_csv(os.path.join(DATA_DIR, "Train.csv"), engine="python")
val_df   = pd.read_csv(os.path.join(DATA_DIR, "Val.csv"),   engine="python")
test_df  = pd.read_csv(os.path.join(DATA_DIR, "Test.csv"),  engine="python")

# ---- For a quick dry-run (CPU / limited time), uncomment to train on a subset: ----
# train_df = train_df.sample(2000, random_state=SEED).reset_index(drop=True)
NUM_EPOCHS = 2

print("Train:", train_df.shape, "| Val:", val_df.shape, "| Test:", test_df.shape)
train_df.head(3)

In [ ]:
# Basic cleaning: drop rows missing source/target, strip whitespace
for df in (train_df, val_df):
    df.dropna(subset=["SOURCE_TEXT", "TARGET_TEXT"], inplace=True)
    df["SOURCE_TEXT"] = df["SOURCE_TEXT"].astype(str).str.strip()
    df["TARGET_TEXT"] = df["TARGET_TEXT"].astype(str).str.strip()
test_df["SOURCE_TEXT"] = test_df["SOURCE_TEXT"].astype(str).str.strip()

print("Language pairs in train (source -> target):")
print(train_df.groupby(["SOURCE_LANG", "TARGET_LANG"]).size())
print("\nExample pair:")
print("SOURCE:", train_df.iloc[0]["SOURCE_TEXT"])
print("TARGET:", train_df.iloc[0]["TARGET_TEXT"])

## 3. Tokenizer, model, and the task prefix

T5 models expect a text-to-text task. We prepend a short instruction such as
`translate French to Havu: ` to each source sentence. The prefix is built per row from
`SOURCE_LANG` / `TARGET_LANG`, so one model learns all 10 directions (French->language and
language->French) from the same code.

In [ ]:
# The five target languages plus French. Codes must match the CSV columns.
LANG_NAMES = {"fr": "French", "hav": "Havu", "ngc": "Ngombe",
              "shr": "Shi", "tap": "Taabwa", "yns": "Yansi"}

def lang_name(code):
    return LANG_NAMES.get(str(code).lower(), str(code))

def make_prefix(src, tgt):
    return "translate " + lang_name(src) + " to " + lang_name(tgt) + ": "

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
print("Loaded", MODEL_NAME, "|", round(sum(p.numel() for p in model.parameters()) / 1e6, 1), "M params")
print("Prefix examples:", repr(make_prefix("fr", "hav")), "|", repr(make_prefix("shr", "fr")))

## 4. Build and tokenize the datasets

In [ ]:
def to_dataset(df, has_target=True):
    cols = {"id":       df["ID"].tolist(),
            "src_lang": df["SOURCE_LANG"].tolist(),
            "tgt_lang": df["TARGET_LANG"].tolist(),
            "source":   df["SOURCE_TEXT"].tolist()}
    if has_target:
        cols["target"] = df["TARGET_TEXT"].tolist()
    return Dataset.from_dict(cols)

train_ds = to_dataset(train_df)
val_ds   = to_dataset(val_df)

def preprocess(batch):
    inputs = [make_prefix(sl, tl) + s
              for sl, tl, s in zip(batch["src_lang"], batch["tgt_lang"], batch["source"])]
    model_inputs = tokenizer(inputs, max_length=MAX_INPUT_LENGTH, truncation=True)
    # tokenize the targets as labels (text_target handles the decoder side correctly)
    labels = tokenizer(text_target=batch["target"], max_length=MAX_TARGET_LENGTH, truncation=True)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

train_tok = train_ds.map(preprocess, batched=True, remove_columns=train_ds.column_names)
val_tok   = val_ds.map(preprocess,   batched=True, remove_columns=val_ds.column_names)
train_tok

## 5. Evaluation metric

We report the **ROUGE F-measure** (ROUGE-1, ROUGE-2 and ROUGE-L), averaged over the
evaluation set; `rougeL` selects the best checkpoint, which is also the **official Zindi score** for this competition (whitespace-tokenized, over all 10 directions) - ROUGE is used here as a
convenient local training signal, not the leaderboard metric.

In [ ]:
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model, label_pad_token_id=-100)

_rouge = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=False)

def rouge_fmeasure(preds, refs):
    totals = {"rouge1": 0.0, "rouge2": 0.0, "rougeL": 0.0}
    n = max(len(preds), 1)
    for p, r in zip(preds, refs):
        sc = _rouge.score(r, p)                 # score(reference, prediction)
        for k in totals:
            totals[k] += sc[k].fmeasure         # F-measure only
    return {k: round(100.0 * v / n, 2) for k, v in totals.items()}

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]
    preds  = np.where(preds  != -100, preds,  tokenizer.pad_token_id)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_preds  = [p.strip() for p in tokenizer.batch_decode(preds,  skip_special_tokens=True)]
    decoded_labels = [l.strip() for l in tokenizer.batch_decode(labels, skip_special_tokens=True)]
    return rouge_fmeasure(decoded_preds, decoded_labels)

## 6. Fine-tune

`fp16` is left **off** (with `bf16` when the GPU supports it): the mT5 / ByT5 variants are
numerically unstable in fp16 and produce `NaN` losses, so this keeps every base model safe.
`t5-base` trains fine either way.

In [ ]:
args = Seq2SeqTrainingArguments(
    output_dir=OUT_DIR,
    eval_strategy="epoch",        # older transformers: rename to evaluation_strategy
    save_strategy="epoch",
    save_total_limit=1,
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    weight_decay=0.01,
    num_train_epochs=NUM_EPOCHS,
    predict_with_generate=True,
    generation_max_length=MAX_TARGET_LENGTH,
    generation_num_beams=NUM_BEAMS,
    fp16=False,                   # keep OFF for mT5/ByT5 (fp16 -> NaN); t5-base is fine either way
    bf16=(DEVICE == "cuda" and torch.cuda.is_bf16_supported()),
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="rougeL",
    greater_is_better=True,
    report_to="none",
    seed=SEED,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

In [ ]:
val_metrics = trainer.evaluate()
print("Validation ROUGE F-measure:",
      {k: val_metrics.get("eval_" + k) for k in ("rouge1", "rouge2", "rougeL")})

## 7. Translate the test set and write the submission

We translate every row of `Test.csv` (each with its own source/target language) and write the
submission directly from it - same rows and IDs - in the required columns
`ID, SOURCE_LANG, SOURCE_TEXT, TARGET_LANG, CANDIDATE_TEXT`. No row may be left blank: empty
predictions score 0, so any gap is filled with a placeholder. Fields with commas are
automatically wrapped in double quotes by `to_csv`.

In [ ]:
model.eval()

@torch.no_grad()
def translate(sources, src_langs, tgt_langs, batch_size=32):
    out = []
    for i in range(0, len(sources), batch_size):
        chunk_src = sources[i:i + batch_size]
        chunk_sl  = src_langs[i:i + batch_size]
        chunk_tl  = tgt_langs[i:i + batch_size]
        inputs = [make_prefix(sl, tl) + s for sl, tl, s in zip(chunk_sl, chunk_tl, chunk_src)]
        enc = tokenizer(inputs, return_tensors="pt", padding=True, truncation=True,
                        max_length=MAX_INPUT_LENGTH).to(model.device)
        gen = model.generate(**enc, max_new_tokens=MAX_TARGET_LENGTH, num_beams=NUM_BEAMS)
        out.extend(tokenizer.batch_decode(gen, skip_special_tokens=True))
    return [o.strip() for o in out]

test_df["PREDICTION"] = translate(test_df["SOURCE_TEXT"].tolist(),
                                  test_df["SOURCE_LANG"].tolist(),
                                  test_df["TARGET_LANG"].tolist())
test_df[["ID", "SOURCE_LANG", "TARGET_LANG", "SOURCE_TEXT", "PREDICTION"]].head()

In [ ]:
# Build the submission straight from Test.csv (same rows/IDs) in the required column order.
submission = test_df[["ID", "SOURCE_LANG", "SOURCE_TEXT", "TARGET_LANG"]].copy()
submission["CANDIDATE_TEXT"] = test_df["PREDICTION"].values

# No blank predictions (empty rows score 0): fill any gaps with a placeholder token.
blank = submission["CANDIDATE_TEXT"].fillna("").astype(str).str.strip() == ""
if blank.any():
    print("WARNING:", int(blank.sum()), "rows had no prediction - filling with a placeholder.")
    submission.loc[blank, "CANDIDATE_TEXT"] = "?"

submission.to_csv(SUBMISSION_PATH, index=False)
print("Saved", SUBMISSION_PATH, "| shape:", submission.shape)
submission.head()

## 8. How to improve your score

The baseline above is intentionally simple. Ideas, roughly in order of effort/payoff:

- **Train all 10 directions together (already set up).** Shared training lets high-resource
  languages lift the low-resource ones - especially **Shi -> Havu**, which are close relatives.
- **Train longer / bigger.** Raise `NUM_EPOCHS`, or switch `MODEL_NAME` to `google/mt5-base`
  (needs more GPU). More epochs help a lot on small data.
- **Try stronger multilingual MT bases.** `google/mt5-*`, `google/byt5-small` (byte-level, great
  for Bantu morphology), or dedicated MT models **NLLB-200** / **M2M-100**. These languages may
  lack a native NLLB tag - add a new language token or reuse a related Bantu language's tag.
- **Tokenizer design.** Subword vocabularies trained mostly on European text fragment Bantu
  morphology badly. Consider a tokenizer-free model (ByT5), training a SentencePiece tokenizer on
  this corpus, or subword regularization - better segmentation often helps more than a bigger model.
- **Clean the data.** Remove near-duplicate verses, fix encoding issues, drop misaligned pairs,
  normalize punctuation. Quality beats quantity here.
- **Decoding.** Increase `NUM_BEAMS` (e.g. 5-8), tune `length_penalty`, or try
  `no_repeat_ngram_size=3` to curb repetition.
- **Back-translation (from the provided data only).** Train a language->French direction,
  translate in-domain source text, and add the synthetic pairs - using only this challenge's data.
- **Watch the metrics.** Local selection uses the **ROUGE F-measure** (`rougeL`); the leaderboard
  uses **`0.1 * BLEU`**. Track both as you iterate.

**Challenge rules to respect:** teams up to 3; up to 5 submissions/day and 35 total; **use only
the provided data** - no external corpora, and never use other translations of the test verses
(e.g. published Bibles or web scrapes), which counts as leakage and leads to disqualification;
set random seeds so your score reproduces.

Good luck!